# GRPO training on a Hugging Face GPU Job

## What you set up (only three things to remember)

1. **`GITHUB_PUBLIC_CLONE_URL`** + **`GIT_CLONE_BRANCH`** — Single branch clone: `git clone --depth 1 --single-branch --branch <branch> <url>`. Set the branch to the one you are developing (not necessarily `main`).
2. **`OPENENV_BASE_URL`** — Your **Hugging Face Space** where the OpenEnv server runs (for training rewards). Not used for `git clone`.
3. **Hugging Face login** — For starting the paid Job and for **pushing the trained adapter** to the Hub.

Then the Job `cd`s to **`/tmp/lh/learn_handwriting`** (repo root + that folder), installs deps, and runs `datagen_sft/grpo_train.py`.

Default GPU in the config cell: **L40S (48GB)**. You need HF Pro/Team for Jobs and credits.

In [ ]:
%pip install -q -U "huggingface_hub>=0.28.0"

In [ ]:
import os
import time

from huggingface_hub import get_token, login, run_job, inspect_job, fetch_job_logs

## 1. Login

Uses `HF_TOKEN` from the environment if set (e.g. Colab secrets / GitHub Actions). Otherwise opens the browser or use `getpass` below.

In [ ]:
if not get_token():
    login(add_to_git_credential=False)
else:
    print("Already logged in (HF_TOKEN / cache).")

### Smoke test (optional)

Run this **after login** to confirm Jobs work with a **cheap CPU** job (`python:3.12` + a one-liner). Check the printed Job URL for logs. Skip if you are confident.

In [ ]:
import time

smoke = run_job(
    image="python:3.12",
    command=[
        "python",
        "-c",
        "import sys, platform; print('Hello from Hugging Face Jobs'); print('python', sys.version.split()[0], platform.node())",
    ],
    flavor="cpu-basic",
    timeout="5m",
)
print("Smoke test job:", smoke)
if getattr(smoke, "url", None):
    print("Open logs:", smoke.url)
smoke_id = getattr(smoke, "id", None)
if smoke_id:
    for _ in range(30):
        st = inspect_job(job_id=smoke_id).status
        if getattr(st, "stage", str(st)) in ("COMPLETED", "ERROR", "CANCELLED"):
            break
        time.sleep(2)
    print("--- log tail ---")
    for line in fetch_job_logs(job_id=smoke_id):
        print(line, end="")

## 2. Job configuration

Edit **`GITHUB_PUBLIC_CLONE_URL`** and **`GIT_CLONE_BRANCH`** (the branch that has your `learn_handwriting/` tree). Then run the launch cell below.

In [ ]:
# Public clone URL. Default matches https://github.com/radharamanaa/OpenEnv-Learn-handwriting (learn_handwriting/ at repo root)
GITHUB_PUBLIC_CLONE_URL = "https://github.com/radharamanaa/OpenEnv-Learn-handwriting.git"
GIT_CLONE_BRANCH = "stage2/long_planning_english"  # must match the branch you pushed to origin (see: git branch --show-current)

HF_USERNAME = "abhijeetmishra101"  # Hugging Face org/user for where the adapter is pushed
OPENENV_BASE_URL = "https://abhijeetmishra101-learn-handwriting.hf.space"  # your Space (OpenEnv), not GitHub

MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"
HUB_MODEL_ID = f"{HF_USERNAME}/Qwen2.5-7B-Handwriting-GRPO"

JOB_IMAGE = "pytorch/pytorch:2.6.0-cuda12.4-cudnn9-devel"
JOB_FLAVOR = "l40sx1"
JOB_TIMEOUT = "8h"

## 3. Launch training Job

Clones **only** `GIT_CLONE_BRANCH` with `--single-branch` (not necessarily `main`).  
Passes **`HF_TOKEN`** so training can **push the adapter to Hugging Face**.

In [ ]:
token = get_token()
if not token:
    raise RuntimeError("No HF token. Run the login cell or set HF_TOKEN.")
if "OWNER/REPO" in GITHUB_PUBLIC_CLONE_URL or GITHUB_PUBLIC_CLONE_URL.count("github.com") == 0:
    raise ValueError(
        'Set GITHUB_PUBLIC_CLONE_URL in the config cell to your *public* repo, e.g. https://github.com/yourname/yourname.git (copy from the green Code button on GitHub)'
    )
print("Will git clone:", GITHUB_PUBLIC_CLONE_URL, "branch:", GIT_CLONE_BRANCH)

remote_script = f"""
set -euo pipefail
export DEBIAN_FRONTEND=noninteractive
apt-get update -qq && apt-get install -y -qq --no-install-recommends git ca-certificates
export PIP_DISABLE_PIP_VERSION_CHECK=1
pip install -q -U pip wheel setuptools
git clone --depth 1 --single-branch --branch "${{GIT_CLONE_BRANCH}}" "${{GITHUB_PUBLIC_CLONE_URL}}" /tmp/lh
REPO_ROOT=/tmp/lh/learn_handwriting
if [ ! -f "$REPO_ROOT/datagen_sft/grpo_train.py" ]; then
  echo "ERROR: expected $REPO_ROOT/datagen_sft/grpo_train.py. We use: git clone ... /tmp/lh, so the repo root is /tmp/lh (not /tmp/lh/Repo-Name). Put learn_handwriting/ at the root of the GitHub repo." >&2
  ls -la /tmp/lh >&2 || true
  exit 1
fi
cd "$REPO_ROOT"
export PIP_ROOT_USER_ACTION=ignore
pip install -q -e "."
pip install -q "torch" "transformers>=4.44.0" "trl>=1.0.0,<2.0.0" "peft" "accelerate" "datasets" "bitsandbytes"
export PYTHONPATH="$REPO_ROOT"
export OPENENV_BASE_URL="${{OPENENV_BASE_URL}}"
export PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True
python datagen_sft/grpo_train.py \\
  --model "${{MODEL_NAME}}" \\
  --hub_model_id "${{HUB_MODEL_ID}}" \\
  --output_dir "$REPO_ROOT/outputs/grpo-handwriting"
echo "Training job script finished."
"""

job = run_job(
    image=JOB_IMAGE,
    command=["bash", "-c", remote_script],
    flavor=JOB_FLAVOR,
    timeout=JOB_TIMEOUT,
    secrets={"HF_TOKEN": token},
    env={
        "GITHUB_PUBLIC_CLONE_URL": GITHUB_PUBLIC_CLONE_URL,
        "GIT_CLONE_BRANCH": GIT_CLONE_BRANCH,
        "OPENENV_BASE_URL": OPENENV_BASE_URL,
        "HUB_MODEL_ID": HUB_MODEL_ID,
        "MODEL_NAME": MODEL_NAME,
    },
)

print("Job:", job)
if getattr(job, "url", None):
    print("URL:", job.url)

## 4. (Optional) Stream logs

Uncomment and set `job_id` from the printed job object if your `huggingface_hub` version returns an id field.

In [ ]:
job_id = getattr(job, "id", None)
if job_id:
    for _ in range(60):
        info = inspect_job(job_id=job_id)
        st = info.status
        print(time.strftime("%H:%M:%S"), st)
        if getattr(st, "stage", str(st)) in ("COMPLETED", "ERROR", "CANCELLED"):
            break
        time.sleep(20)
    for line in fetch_job_logs(job_id=job_id):
        print(line, end="")
else:
    print("No job id on result; open the Job URL in the browser for logs.")